In [0]:

from pyspark.sql.functions import col, count, sum as spark_sum, date_trunc

CATALOG = "workspace"
TARGET_SCHEMA = "insurance_lakehouse"

SILVER_POLICY_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.silver_policy"
SILVER_CLAIMS_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.silver_claims"

GOLD_CLAIM_ENRICHED_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.gold_claim_enriched"
GOLD_CLAIM_SUMMARY_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.gold_claim_summary"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {TARGET_SCHEMA}")

DataFrame[]

In [0]:

def has_col(df, c):
    return c in df.columns

policy_df = spark.table(SILVER_POLICY_TABLE)
claims_df = spark.table(SILVER_CLAIMS_TABLE)

print("Policy columns:", policy_df.columns)
print("Claims columns:", claims_df.columns)

Policy columns: ['policy_id', 'customer_id', 'policy_number', 'policy_type', 'product_code', 'policy_status', 'issue_date', 'policy_start_date', 'policy_end_date', 'premium_amount', 'premium_frequency', 'sum_assured', 'payment_method', 'agent_id', 'branch_id', 'state', 'customer_dob', 'customer_gender', 'annual_income', 'nominee_relation', 'created_ts', 'updated_ts', 'coverage_amount', 'source_file', 'ingestion_ts', 'snapshot_date']
Claims columns: ['claim_id', 'policy_id', 'claim_number', 'claim_type', 'claim_status', 'claim_date', 'claim_reported_date', 'claim_amount', 'approved_amount', 'settlement_amount', 'rejection_reason', 'hospital_id', 'diagnosis_code', 'death_cause_code', 'claim_channel', 'state', 'created_ts', 'updated_ts', 'source_file', 'ingestion_ts', 'snapshot_date']


In [0]:

# ----------------------------
# GOLD 1: Enriched claim detail
# Join claims with policy
# ----------------------------
joined_df = claims_df.alias("c").join(
    policy_df.alias("p"),
    on="policy_id",
    how="left"
)

select_exprs = []

# Claims columns
for c in [
    "claim_id",
    "policy_id",
    "claim_number",
    "claim_type",
    "claim_status",
    "claim_date",
    "updated_ts",
    "snapshot_date"
]:
    if has_col(claims_df, c):
        select_exprs.append(col(f"c.{c}").alias(c))

# Policy columns
for c in [
    "policy_number",
    "customer_id",
    "customer_name",
    "policy_type",
    "policy_status",
    "premium_amount",
    "coverage_amount",
    "policy_start_date",
    "policy_end_date"
]:
    if has_col(policy_df, c):
        select_exprs.append(col(f"p.{c}").alias(c))

gold_claim_enriched_df = joined_df.select(*select_exprs)

(
    gold_claim_enriched_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_CLAIM_ENRICHED_TABLE)
)

print(f"Created/updated {GOLD_CLAIM_ENRICHED_TABLE}")
display(spark.table(GOLD_CLAIM_ENRICHED_TABLE))

Created/updated workspace.insurance_lakehouse.gold_claim_enriched


claim_id,policy_id,claim_number,claim_type,claim_status,claim_date,updated_ts,snapshot_date,policy_number,customer_id,policy_type,policy_status,premium_amount,coverage_amount,policy_start_date,policy_end_date
CLM000003738,P00241349,CLNO200003737,ACCIDENT,APPROVED,2026-01-23,2021-06-07T02:36:15.000Z,2026-03-26,POL100241348,CUST00241349,ENDOWMENT,ACTIVE,6025.77,388786.82,2024-10-02,2035-08-28
CLM000007216,P00036285,CLNO200007215,SURGERY,APPROVED,2022-09-03,2023-07-21T00:17:21.000Z,2026-03-26,POL100036284,CUST00036285,WHOLE,ACTIVE,5456.1,150503.12,2020-06-28,2047-12-05
CLM000011175,P00081286,CLNO200011174,HOSPITALIZATION,APPROVED,2023-07-08,2019-01-07T05:17:24.000Z,2026-03-26,POL100081285,CUST00081286,TERM,ACTIVE,2214.2,48203.68,2023-05-05,2053-02-04
CLM000011421,P00037139,CLNO200011420,HOSPITALIZATION,REJECTED,2021-10-31,2025-09-23T14:01:52.000Z,2026-03-26,POL100037138,CUST00037139,TERM,ACTIVE,1661.2,45301.87,2019-02-04,2045-07-07
CLM000013731,P00238749,CLNO200013730,DEATH,PENDING,2020-12-20,2026-01-14T22:50:57.000Z,2026-03-26,POL100238748,CUST00238749,ENDOWMENT,ACTIVE,8907.74,293466.06,2020-05-18,2044-10-20
CLM000014504,P00257450,CLNO200014503,SURGERY,UNDER_REVIEW,2025-09-12,2024-07-12T01:02:29.000Z,2026-03-26,POL100257449,CUST00257450,ULIP,ACTIVE,5049.9,315608.91,2018-12-31,2044-05-24
CLM000016976,P00200294,CLNO200016975,SURGERY,APPROVED,2023-06-16,2025-10-09T06:33:32.000Z,2026-03-26,POL100200293,CUST00200294,TERM,ACTIVE,13337.34,1047841.15,2021-09-02,2043-01-12
CLM000018697,P00157428,CLNO200018696,DISABILITY,UNDER_REVIEW,2023-09-18,2025-07-11T02:17:46.000Z,2026-03-26,POL100157427,CUST00157428,TERM,CLOSED,10011.44,434106.25,2017-10-17,2041-08-08
CLM000019670,P00178441,CLNO200019669,ACCIDENT,PENDING,2020-12-20,2023-06-23T17:21:38.000Z,2026-03-26,POL100178440,CUST00178441,ULIP,ACTIVE,7635.46,266285.06,2018-08-03,2035-07-28
CLM000020144,P00222730,CLNO200020143,HOSPITALIZATION,UNDER_REVIEW,2018-01-15,2024-02-22T20:00:42.000Z,2026-03-26,POL100222729,CUST00222730,ENDOWMENT,ACTIVE,7677.65,485315.19,2016-11-07,2043-02-24


In [0]:
from pyspark.sql.functions import col, count, sum as spark_sum, date_trunc, to_date

claims_for_summary_df = claims_df.withColumn(
    "claim_month",
    to_date(date_trunc("month", col("claim_date")))
)

gold_claim_summary_df = (
    claims_for_summary_df
    .groupBy("claim_month", "claim_type", "claim_status")
    .agg(
        count("*").alias("claim_count"),
        spark_sum(col("claim_amount").cast("decimal(18,2)")).alias("total_claim_amount")
    )
)

(
    gold_claim_summary_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_CLAIM_SUMMARY_TABLE)
)

display(spark.table(GOLD_CLAIM_SUMMARY_TABLE))

claim_month,claim_type,claim_status,claim_count,total_claim_amount
2025-02-01,DISABILITY,UNDER_REVIEW,240,29371368.28
2026-01-01,DISABILITY,PENDING,659,84582528.98
2024-10-01,SURGERY,PENDING,298,37362768.17
2025-10-01,SURGERY,APPROVED,1154,145940596.07
2024-07-01,ACCIDENT,APPROVED,817,103846918.75
2023-12-01,DISABILITY,APPROVED,510,65554790.46
2020-12-01,SURGERY,APPROVED,200,23858936.41
2024-01-01,DISABILITY,APPROVED,522,66838606.79
2025-11-01,HOSPITALIZATION,REJECTED,885,112549802.82
2024-12-01,DISABILITY,APPROVED,719,91526399.79
